In [ ]:
import numpy as np
import pandas as pd
import os
import glob
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LogNorm
from scipy.stats import pearsonr, spearmanr, brunnermunzel
from statsmodels.stats.multitest import multipletests
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
import difflib
from scipy.stats import binned_statistic, binned_statistic_2d
# function to calculate standard error of the mean
se = lambda x : np.std(x)/np.sqrt(x.size)
se.__name__ = 'SE'

def get_overlap(s1, s2):
    s = difflib.SequenceMatcher(None, s1, s2)
    pos_a, pos_b, size = s.find_longest_match(0, len(s1), 0, len(s2)) 
    sequence = ['-']*len(s1)
    sequence[pos_a:pos_a+size] = list(s1)[pos_a:pos_a+size]
    return ''.join(sequence),size

def format_scientific(x):
    a = x.split('e')[0]
    b = int(x.split('e')[1])
    return a+'\\times'+'10^{'+f'{b:d}'+'}'

def bootstrap_cohens_d(a, b, n_bootstrap=10000, subsample=False):
    a, b = np.asarray(a), np.asarray(b)
    cohens_d = (a.mean() - b.mean()) / np.sqrt(0.5 * (a.std()**2 + b.std()**2))
    a_size = a.size
    b_size = a_size if subsample else b.size
    d_boot = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        a_p = np.random.choice(a, size=a_size, replace=True)
        b_p = np.random.choice(b, size=b_size, replace=True)
        d_boot[i] = (a_p.mean() - b_p.mean()) / np.sqrt(0.5 * (a_p.std()**2 + b_p.std()**2))
    return cohens_d, d_boot.std()

hexcolors = ['#EE7733', '#0077BB', '#33BBEE', '#EE3377', '#CC3311', '#009988', '#BBBBBB']
colornames = ['orange','blue','cyan','magenta','red','teal','grey']
pt_colors = dict(zip(colornames,hexcolors))

In [ ]:
dfs = []
for batch in range(0, 20091 + 200, 200):
    df = pd.read_csv(f'data/larks_{batch:d}.csv.gz', index_col=0)
    if not df.empty and not df.isna().all().all():
        dfs.append(df)
df_larks = pd.concat(dfs)

df_idrome = pd.read_csv('IDRome_DB_PSpred.csv',index_col=1)

df_larks["N_LARKS"] = pd.to_numeric(df_larks["N_LARKS"], errors="coerce")
match_100 = df_larks.query('f_match==1').groupby('IDR_name').N_LARKS.mean()

common_seq = np.intersect1d(match_100.index,df_idrome.index)

df_idrome.loc[common_seq,'N_LARKS'] = match_100.loc[common_seq]

df_idrome.loc[common_seq,'f_LARKS'] = df_idrome.loc[common_seq,'N_LARKS']/df_idrome.loc[common_seq,'N']

In [ ]:
f = plt.figure(figsize=(4,3))
plt.rc('font', **{'family':'sans-serif', 'size': 10})

feature = 'f_LARKS'

bin_width = .5
bins = np.arange(-12.25, -2.5, bin_width)
x = bins[:-1]+bin_width/2

non_nan_bool = ~df_idrome[feature].isnull()

average = binned_statistic(df_idrome.dG_pred.values[non_nan_bool],
                               df_idrome[feature].values[non_nan_bool],bins=bins)[0]
standard_error = binned_statistic(df_idrome.dG_pred.values[non_nan_bool],
                                  df_idrome[feature].values[non_nan_bool],statistic=se,
                                  bins=bins)[0]
counts = binned_statistic(df_idrome.dG_pred.values[non_nan_bool],
                           df_idrome[feature].values[non_nan_bool],bins=bins,statistic='count')[0]

plt.errorbar(x[counts>2],average[counts>2],yerr=standard_error[counts>2],ms=0,marker='o',lw=0,zorder=0,
         elinewidth=.5,capsize=1.5,capthick=.5,color='k')

plt.ylabel('Mean $f_\mathrm{LARKS}$')

plt.xlabel('$\Delta G$  ($k_BT$)')

im = plt.scatter(x[counts>2],average[counts>2],marker='o',
                c=counts[counts>2],vmin=3,vmax=counts.max(),cmap=plt.cm.viridis_r,s=20,
                edgecolor='k',lw=.2,zorder=10)

plt.ylabel('Proteome-wide mean $f_\mathrm{LARKS}$')

plt.xlabel(r"$\Delta G_\mathrm{pred}$ ($k_B T$)")

divider = make_axes_locatable(plt.gca())
cax = divider.new_vertical(size="{:.1f}%".format(5), pad=.1)
f.add_axes(cax)
cb = f.colorbar(im, cax=cax, orientation='horizontal',ticks=[3,25,50,75,100],
            label='Number of IDRs')
cax.xaxis.set_label_position('top'); cax.xaxis.set_ticks_position('top')
plt.tight_layout()
plt.savefig('Fig6k.pdf')
plt.show()

In [ ]:
fig, (ax1,ax3) = plt.subplots(1,2,figsize=(7, 3.4))

feature = "f_LARKS"
df = df_idrome[["dG_pred", feature]].dropna()
x, y = df["dG_pred"].to_numpy(), df[feature].to_numpy()

bin_width = .5
bins = np.arange(-12.25, -2.5, bin_width)
thresholds = bins[:-1]+bin_width/2
min_n, n_boot = 100, 10000
rng = np.random.default_rng(1)

rho = np.full(len(thresholds), np.nan)
rho_lo = np.full(len(thresholds), np.nan)
rho_hi = np.full(len(thresholds), np.nan)
nvals = np.zeros(len(thresholds), dtype=int)

for i, t in enumerate(thresholds):
    m = x < t
    nvals[i] = m.sum()
    if nvals[i] >= min_n:
        rho[i], _ = spearmanr(x[m], y[m])
        boot = [spearmanr(x[m][idx], y[m][idx])[0]
                for idx in (rng.integers(0, nvals[i], nvals[i]) for _ in range(n_boot))]
        rho_lo[i], rho_hi[i] = np.nanpercentile(boot, [2.5, 97.5])

rho_err = np.vstack([rho - rho_lo, rho_hi - rho])

ax2 = ax1.twinx()

ax1.errorbar(thresholds, rho, yerr=rho_err,
             marker="o", color="k",
             ms=5, lw=1, elinewidth=.5, capsize=2, capthick=.5)

ax2.plot(thresholds[nvals>=min_n], nvals[nvals>=min_n], marker="o",
         color=pt_colors['blue'], ms=4, lw=1)

ax1.set_xlabel(r"$\Delta G_\mathrm{pred}$ threshold ($k_B T$)")
ax1.set_ylabel(r"Spearman $\rho$")
ax2.set_ylabel("Number of IDRs", color=pt_colors['blue'])
ax1.tick_params(axis="y", labelcolor="k")
ax2.tick_params(axis="y", labelcolor=pt_colors['blue'])

###

feature = "f_LARKS"
threshold = -4
n_bootstrap = 10000

bin_width = 0.02
bins = np.arange(0, df_idrome[feature].max() + bin_width, bin_width)
x = bins[:-1] + bin_width / 2

df = df_idrome[["dG_pred", feature]].dropna()
bool_term = (df.dG_pred < threshold).values

a = df[bool_term][feature].values
b = df[~bool_term][feature].values

print(df[~bool_term].shape[0])
N_IDRs = df[bool_term].shape[0]
h = np.histogram(a, bins=bins, density=True)[0]

ax3.hist(b, bins, density=True, histtype="step", color="k", zorder=0)

alternative = "greater"
color = pt_colors["blue"]
if a.mean() < b.mean():
    alternative = "less"
    color = pt_colors["orange"]

W, p = brunnermunzel(a, b, alternative=alternative)
cohens_d, cohens_d_se = bootstrap_cohens_d(a, b, n_bootstrap=n_bootstrap)

print(f"dG_pred < {threshold}: Cohen's d = {cohens_d:.2f} ± {cohens_d_se:.2f}, Brunner-Munzel p = {p:.2e}")

ax3.bar(
    x, h, alpha=.7, width=bin_width, color=color,
    label=r"Cohen's $d$:"
          + f"\n{cohens_d:.2f}±{cohens_d_se:.2f}"
          + f"\n$P="+format_scientific(f"{p:.0e}")+'$'
)

ax3.set_xlabel(r"$f_\mathrm{LARKS}$")
ax3.set_ylabel(r"$P(f_\mathrm{LARKS})$")
ax3.set_yscale("log")

legend = ax3.legend(handlelength=0, frameon=False, borderpad=0, 
                   title=rf"{N_IDRs:,d} IDRs with"+"\n"+f"$\Delta G_\mathrm{{pred}} < {threshold}\;k_B T$")
legend.get_title().set_ha("center")

ax1.annotate('a',(-.3,.95),xycoords='axes fraction',fontsize=12,fontweight='bold')
ax3.annotate('b',(-.3,.95),xycoords='axes fraction',fontsize=12,fontweight='bold')

plt.tight_layout()
plt.savefig('FigS13.pdf')
plt.show()